In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.helperFunctions import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [23]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 12 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [11]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Tyrese Maxey,Over,32.5,-137,2025-12-03,2025-12-02T18:50:30Z
1,PrizePicks,player_points,Tyrese Maxey,Under,32.5,-137,2025-12-03,2025-12-02T18:50:30Z
2,PrizePicks,player_points,C.J. McCollum,Over,20.5,-137,2025-12-03,2025-12-02T18:50:30Z
3,PrizePicks,player_points,C.J. McCollum,Under,20.5,-137,2025-12-03,2025-12-02T18:50:30Z
4,PrizePicks,player_points,Quentin Grimes,Over,17.5,-137,2025-12-03,2025-12-02T18:50:30Z


In [12]:
from PRODUCTION.featureEngine.feature_engine import FeatureEngine

# Initialize FeatureEngine with NGBOOST model for points prediction
engine = FeatureEngine({
    "min_model": "../MODELS/SAVED_MODELS/min_model.pkl",
    "usg_model": "../MODELS/SAVED_MODELS/usg_model.pkl",
    "ngboost_model_paths": {
        "mean_model": "../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl",
        "variance_model": "../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl",
        "calibration_factor": "../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl",
        "calibration_params": "../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_PARAMS_PRODUCTION.pkl",  # Add this
        "features": "../MODELS/SAVED_MODELS/pts_features.pkl"
    }
})

# Test prediction for a single player
result = engine.project_player(
    player_name="Draymond Green",
    data=s26,
    date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

print("Prediction Result:")
print(f"  Predicted Minutes: {result['predicted_minutes']:.2f}")
print(f"  Predicted Usage: {result['predicted_usage']:.3f}")
print(f"  Predicted Points: {result['predicted_points']:.2f}")
print(f"\nFull result: {result}")

Prediction Result:
  Predicted Minutes: 35.96
  Predicted Usage: 0.185
  Predicted Points: 18.56

Full result: {'predicted_minutes': 35.955047607421875, 'predicted_usage': 0.1851179599761963, 'predicted_points': 18.56032304566646}


In [13]:
# In your LIVE.ipynb notebook
from scipy.stats import truncnorm

pred_data = get_cached_prediction_v2(
    player_name="Tyrese Maxey",
    data=s26,
    engine=engine,
    current_date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

mu = pred_data['prediction']
sigma = pred_data['sigma']
variance = pred_data['variance']

# Calculate 95% CI using truncated normal (can't go below 0)
lower_bound = 0
upper_bound = 50  # Reasonable max for any player

# Convert to standardized bounds
a = (lower_bound - mu) / sigma
b = (upper_bound - mu) / sigma

# Create truncated distribution
dist = truncnorm(a, b, loc=mu, scale=sigma)

# Get 95% confidence interval (2.5% and 97.5% percentiles)
ci_lower = dist.ppf(0.025)
ci_upper = dist.ppf(0.975)

print(f"Predicted Points (mu): {mu:.2f}")
print(f"Sigma (std dev): {sigma:.2f}")
print(f"Variance: {variance:.2f}")
print(f"95% Confidence Interval: [{ci_lower:.1f}, {ci_upper:.1f}]")

Predicted Points (mu): 29.69
Sigma (std dev): 7.01
Variance: 49.20
95% Confidence Interval: [15.9, 43.2]


## Top EVs for 2 leg bets

### Underdog picks

In [14]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 65 players...
Processing 62 players...
Generated 1756 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
1639,Jaden McDaniels,Aaron Wiggins,13.5,10.5,-112,-105,22.08,18.89,0.890,0.875,over,over,1,129.02,0.645,High,High
149,VJ Edgecombe,Zion Williamson,13.5,22.5,-105,-122,21.49,15.67,0.855,0.870,over,under,1,118.80,0.594,High,High
1266,Jordan Walsh,Jaylin Williams,5.5,4.5,-104,-137,10.68,9.00,0.852,0.842,over,over,1,111.04,0.555,Low,Low
265,Cam Whitmore,Trey Murphy III,13.5,19.5,-102,-103,9.98,14.09,0.822,0.821,under,under,0,98.36,0.492,Low,High
431,Marvin Bagley III,Derik Queen,13.5,12.5,100,105,10.18,18.81,0.803,0.798,under,over,0,88.53,0.443,Low,High


### Prizepicks picks

In [15]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24)]

prizepicksPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 75 players...
Processing 71 players...
Generated 2301 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
2244,Mike Conley,Aaron Wiggins,3.5,10.5,-137,-105,8.03,18.89,over,over,0.944,0.875,0.8100,0.380,0.376,0.527,142.99,0.715,1,2.55,6.92,Low,High,"(3.0, 13.0)","(5.3, 32.5)",0.05,0,143.0
308,VJ Edgecombe,Jaden McDaniels,13.5,13.5,-105,-112,21.49,22.08,over,over,0.855,0.890,0.7460,0.356,0.374,0.487,123.79,0.619,1,7.09,6.60,High,High,"(7.6, 35.4)","(9.1, 35.0)",0.05,0,123.8
1526,Jordan Walsh,Anthony Edwards,5.5,23.5,-104,-137,10.68,31.88,over,over,0.852,0.872,0.7284,0.355,0.308,0.446,118.52,0.593,1,4.65,6.92,Low,High,"(1.6, 19.8)","(18.3, 45.4)",0.05,0,118.5
1046,Gradey Dick,Zion Williamson,5.5,22.5,-135,-122,10.23,15.67,over,under,0.852,0.870,0.7263,0.291,0.334,0.424,117.88,0.589,1,4.14,6.48,Low,High,"(2.1, 18.3)","(3.0, 28.4)",0.05,0,117.9
2159,Donte DiVincenzo,Jaylin Williams,13.5,4.5,105,-137,20.66,9.00,over,over,0.870,0.842,0.7178,0.394,0.278,0.448,115.35,0.577,1,5.93,4.22,Med,Low,"(9.0, 32.3)","(0.7, 17.3)",0.05,0,115.4
1954,Luke Kornet,Trey Murphy III,8.0,19.5,-137,-103,13.61,14.09,over,under,0.823,0.821,0.6621,0.259,0.326,0.382,98.64,0.493,1,5.67,6.37,Med,High,"(2.5, 24.7)","(1.6, 26.6)",0.05,0,98.6
388,Marvin Bagley III,Cason Wallace,13.5,7.5,100,102,10.18,13.58,under,over,0.803,0.818,0.6439,0.315,0.335,0.407,93.18,0.466,0,4.44,6.57,Low,High,"(1.5, 18.9)","(0.7, 26.5)",0.05,0,93.2
2063,Julius Randle,Quinten Post,21.5,8.0,-103,-137,27.64,13.37,over,over,0.793,0.794,0.6171,0.298,0.230,0.337,85.12,0.426,1,6.91,6.27,High,High,"(14.1, 41.2)","(1.1, 25.7)",0.05,0,85.1
437,Bilal Coulibaly,Naz Reid,11.0,13.5,-137,100,17.41,19.38,over,over,0.792,0.788,0.6114,0.228,0.300,0.335,83.41,0.417,1,7.52,6.79,High,High,"(2.7, 32.2)","(6.1, 32.7)",0.05,0,83.4
759,Toumani Camara,Neemias Queta,12.5,9.5,-105,-114,18.61,14.78,over,over,0.777,0.776,0.5909,0.278,0.256,0.330,77.28,0.386,1,7.51,6.59,High,High,"(3.9, 33.3)","(1.9, 27.7)",0.05,0,77.3


## 3 leg parlay

### Underdog picks

In [16]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 65 players...
Processing 62 players...
Generated 30042 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
4296,VJ Edgecombe,Jaden McDaniels,Aaron Wiggins,13.5,13.5,10.5,21.49,22.08,18.89,0.855,0.890,0.875,over,over,over,1,259.87,0.520,High,High,High
26155,Jordan Walsh,Zion Williamson,Jaylin Williams,5.5,22.5,4.5,10.68,15.67,9.00,0.852,0.870,0.842,over,under,over,1,237.21,0.474,Low,High,Low
6244,Cam Whitmore,Toumani Camara,Trey Murphy III,13.5,12.5,19.5,9.98,18.61,14.09,0.822,0.777,0.821,under,over,under,0,183.24,0.366,Low,High,High
10668,Marvin Bagley III,Neemias Queta,Derik Queen,13.5,9.5,12.5,10.18,14.78,18.81,0.803,0.776,0.798,under,over,over,0,168.59,0.337,Low,High,High
5537,Bilal Coulibaly,Jaylen Wells,Julius Randle,11.5,12.5,21.5,17.41,18.26,27.64,0.772,0.768,0.793,over,over,over,1,153.87,0.308,High,High,High


### Prizepicks picks

In [17]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 72 players...
Processing 68 players...
Generated 39554 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
8755,VJ Edgecombe,Jaden McDaniels,Aaron Wiggins,13.5,13.5,10.5,21.49,22.08,18.89,0.855,0.890,0.875,over,over,over,1,259.87,0.520,High,High,High
25283,Gradey Dick,Jordan Walsh,Anthony Edwards,5.5,5.5,23.5,10.23,10.68,31.88,0.852,0.852,0.872,over,over,over,1,241.93,0.484,Low,Low,High
37991,Luke Kornet,Zion Williamson,Cason Wallace,8.0,22.5,7.5,13.61,15.67,13.58,0.823,0.870,0.818,over,under,over,1,216.31,0.433,Med,High,High
10419,Marvin Bagley III,Donte DiVincenzo,Quinten Post,13.5,13.5,8.0,10.18,20.66,13.37,0.803,0.870,0.794,under,over,over,0,199.61,0.399,Low,Med,High
10843,Bilal Coulibaly,Toumani Camara,Trey Murphy III,11.0,12.5,19.5,17.41,18.61,14.09,0.792,0.777,0.821,over,over,under,1,172.97,0.346,High,High,High
